# 01 · Mine related-work sections and citation contexts from arXiv

**Purpose.** Build the datasets every later stage depends on: one row per citing sentence x cited paper, and one row per paper pairing its human-written related-work section with the abstracts of the papers it cites. Also loads Multi-XScience as an external benchmark with the same schema.

| | |
|---|---|
| **Inputs** | the arXiv API and OpenAlex (no local inputs) |
| **Outputs** | `data/citation_contexts.jsonl`, `data/rw_pairs.jsonl`, `data/openalex_cache.json`, `data/multixscience_test.jsonl`, `results/stage1_yield.json`, `results/stage1_dataset_stats.json` |
| **Expected runtime (Colab T4)** | ~45 min at pilot scale (150/50 papers); 2-6 h at full scale (3000/300) |
| **Required secrets** | `OPENALEX_MAILTO` |
| **GPU** | not needed |

Everything is cached under `/content/drive/MyDrive/litreview/` and every stage is
**resumable**: if Colab disconnects, re-run all cells and it continues where it
stopped. Only the cell marked CONFIG needs editing.


### What the two datasets are for

| File | What it holds | Used by |
|---|---|---|
| `citation_contexts.jsonl` | one row per (citing paper, cited paper, sentence that cites it) | the relation classifier (component A, notebook 02) |
| `rw_pairs.jsonl` | a paper's related-work section + the titles/abstracts of the papers it cites | the evaluation benchmark (notebook 06) and the distillation data (05) |
| `openalex_cache.json` | cited title -> OpenAlex metadata | everything |

**Time split.** Train papers come from an older window, test papers from a recent
one. Test papers must be newer than the knowledge cutoff of every model being
evaluated, otherwise a model may have memorised the real related-work section.
The dates used and why are in `docs/leakage.md`.


In [ ]:
# --- Colab setup: Drive, repo, secrets. Safe to re-run. ---
import json, os, subprocess, sys
from pathlib import Path
ON_COLAB = "google.colab" in sys.modules or os.path.exists("/content")

if ON_COLAB:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)

ROOT = "/content/drive/MyDrive/litreview" if ON_COLAB else os.path.abspath("..")
os.makedirs(ROOT, exist_ok=True)
os.environ["LITREVIEW_ROOT"] = ROOT

# Install the project. Two options, in order of preference:
#   1. the repo is already on Drive at MyDrive/litreview/repo  -> editable install
#   2. clone it from git (set REPO_URL below)
REPO = os.path.join(ROOT, "repo")
if ON_COLAB:
    if not os.path.exists(os.path.join(REPO, "pyproject.toml")):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO], check=False)
    if os.path.exists(os.path.join(REPO, "pyproject.toml")):
        subprocess.run([sys.executable, "-m", "pip", "-q", "install", "-e",
                        f"{REPO}[ml,eval]"], check=True)
    else:
        raise SystemExit(
            "Put this repository at MyDrive/litreview/repo (upload or clone it), "
            "or set REPO_URL in this cell. The notebooks deliberately do not "
            "contain a second copy of the code.")

import litreview
from litreview.config import SETTINGS, paths, secret
P = paths().mkdirs()
print("litreview", litreview.__version__ if hasattr(litreview, "__version__") else "", "root:", P.root)


### Config

Start with a small pilot (a few hundred papers) to check yield, then scale up.
Roughly half of arXiv submissions have usable LaTeX with an identifiable
related-work section; the exact figures for this run are printed below.

In [ ]:
# ======================= CONFIG (the only cell to edit) =======================
CATEGORIES   = ["cs.CL", "cs.IR"]                 # NLP + IR: closest to the project's domain
TRAIN_WINDOW = ("202301010000", "202412312359")
TEST_WINDOW  = ("202606010000", "202609302359")   # after the evaluated models' cutoffs
N_TRAIN, N_TEST = 150, 50                         # pilot; scale to ~3000 / 300
N_MULTIXSCIENCE = 300                             # external benchmark examples (0 to skip)
SEED = 0
# ==============================================================================


### The parsing library

The LaTeX miner lives in `litreview.mining.latex_mining` and is unit-tested
offline (`tests/test_latex_mining.py`). It finds the main `.tex` among several
candidates, inlines `\input{}`, strips comments, locates the related-work section
whether it is a `\section`, a `\subsection` or only a run of `\paragraph` blocks,
converts LaTeX to text while keeping citations as `[CITE:key]` markers, and maps
cite keys to titles from `.bbl` (both `\bibitem` and `biblatex` formats), `.bib`,
or a `thebibliography` block inlined in the main file.

In [ ]:
from litreview.mining import latex_mining as lm

# the same self-test the unit tests run, as a smoke check that the install worked
_t = lm.mine_paper({
    "main.tex": r"\documentclass{article}\begin{document}\section{Related Work}"
                r"Pointer networks were used by \citet{see2017}. In contrast, \citep{liu2019} "
                r"fine-tune BERT encoders for summarization.\section{Method}x\end{document}",
    "refs.bib": "@article{see2017,\n title = {Get to the point},\n}\n",
})
assert _t["related_work"]["cited_keys"] == ["liu2019", "see2017"], _t
print("parser OK:", _t["related_work"]["text"])


### Mine

`run_stage1` lists both splits from the arXiv API, downloads and caches every
LaTeX source, parses each paper, resolves cited titles against OpenAlex and writes
the two datasets. Each of those steps is cached, so this cell is safe to
re-run.

arXiv asks for at least three seconds between requests, which is enforced inside
the package. At pilot scale expect roughly 20 minutes of downloading and 20
minutes of OpenAlex resolution.

In [ ]:
from tqdm.auto import tqdm
from litreview.mining.pipeline import run_stage1

report = run_stage1(
    N_TRAIN, N_TEST, CATEGORIES, TRAIN_WINDOW, TEST_WINDOW, seed=SEED,
    fetch_progress=lambda ps, s: tqdm(ps, desc=f"fetch {s}"),
    progress=lambda ps, s: tqdm(ps, desc=f"parse {s}"),
    oa_progress=lambda ts: tqdm(ts, desc="openalex"),
)
print(json.dumps(report, indent=2))


### Multi-XScience

The same task from a published dataset (Lu, Dong and Charlin, EMNLP 2020),
converted into the `rw_pairs` schema with the same filters applied. It is a check
that the evaluation numbers are not an artefact of our own mining.

In [ ]:
from litreview.mining import multixscience

if N_MULTIXSCIENCE:
    mx = multixscience.build(P.data, split="test", limit=N_MULTIXSCIENCE)
    print(json.dumps(mx, indent=2))
    report["multixscience"] = mx
    (P.results / "stage1_yield.json").write_text(json.dumps(report, indent=2))


### Statistics and sanity checks

Read a few samples by eye before moving on: bad parsing here silently poisons
every later stage. The time-split check is recorded to `results/` rather than
asserted, so the report can quote the actual dates.

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, (Path(REPO) / "scripts" / "dataset_stats.py").as_posix()
                if ON_COLAB else "../scripts/dataset_stats.py"], check=False)

import pandas as pd
ctx = pd.read_json(P.data / "citation_contexts.jsonl", lines=True)
rw  = pd.read_json(P.data / "rw_pairs.jsonl", lines=True)
print(ctx.groupby("split").agg(rows=("cite_key", "size"), papers=("citing_id", "nunique"),
                               in_rw=("in_related_work", "mean")).round(2))
rw["n_cited"]  = rw["cited"].str.len()
rw["rw_words"] = rw["rw_text"].str.split().str.len()
print(rw.groupby("split")[["n_cited", "rw_words", "abstract_coverage"]].median())

for _, r in ctx.sample(min(3, len(ctx)), random_state=1).iterrows():
    print("\n---", r.cited_title, "\n", r.context[:400])


In [ ]:
# Leakage check: the newest train paper must predate the oldest test paper.
ts = json.loads((P.results / "stage1_dataset_stats.json").read_text())["time_split"]
print(json.dumps(ts, indent=1))
assert ts["disjoint"], "time split overlap -- do not evaluate on this benchmark"


### Next

**02_relation_labels_and_scibert.ipynb** — define the relation taxonomy, label a
few thousand contexts with the configured LLM, hand-check 200 for agreement, then
fine-tune SciBERT (component A).

`rw_pairs.jsonl` (test split) is the frozen evaluation benchmark. Nothing is tuned
on it.